# Chapter 7: Automating Analysis with Bash Scripting

## 1. Introduction

In previous lessons, you executed commands one by one in the terminal. Now, you will learn to chain them together into powerful, automated scripts. This section focuses on writing robust, repeatable, and scalable bash scripts to automate complex bioinformatics analyses. Automating the analysis of genomic data, such as VCF or FASTQ files, is not just a convenience; it is a scientific necessity. It ensures your results are reproducible, scalable across hundreds of samples, and less prone to human error—all of which are critical for both research and clinical diagnostics.

> **Key Terms:**
> *   **VCF (Variant Call Format):** A standard text file format for storing gene sequence variations. Each line typically represents a specific variant (like a SNP or an indel) found in a sample.
> *   **FASTQ:** A text-based format for storing both a biological sequence (like a DNA read) and its corresponding quality scores. It's a fundamental format in next-generation sequencing.

---

## 2. Key Concepts and Definitions

*   **Bash Script**: A plain text file containing a series of shell commands that are executed sequentially. In bioinformatics, scripts act as the "glue" that connects different analysis tools (e.g., aligners, variant callers) into a single, automated pipeline.
*   **Error Handling**: The process of anticipating, detecting, and resolving errors within a script to ensure it runs reliably. In a clinical genomics setting, robust error handling is non-negotiable; it prevents a script from failing silently or producing incorrect results that could lead to a misdiagnosis.
*   **Batch Processing**: The ability to automate a repetitive task on a large number of files without manual intervention. A medical analogy would be a lab robot that can automatically process hundreds of patient blood samples overnight, ensuring consistency and high throughput.

---

## 3. Main Content

### 3.1 Build Robust Scripts with Error Handling

Make your analysis safer with configurable scripts that handle errors. This example counts variants in a VCF file, exiting safely if the input is missing and correctly processing compressed files.

> **Pro Tip:** Always start your scripts with `set -euo pipefail`. This "fail-fast" approach is a crucial best practice that prevents your script from continuing with unexpected errors, which could lead to corrupted data or incorrect scientific conclusions. It forces the script to stop immediately if a command fails (`-e`), a variable is unset (`-u`), or a command in a pipeline fails (`-o pipefail`).

In [ ]:
%%bash
#!/bin/bash
set -euo pipefail
# --- Configuration ---
# Usage: $0 <variants.vcf[.gz]>
VCF_FILE=${1:?"Error: Missing input file. Usage: $0 <variants.vcf[.gz]>"}
if [ ! -f "${VCF_FILE}" ]; then
echo "Error: Input file not found: ${VCF_FILE}" >&2
exit 1
fi
# --- Workflow ---
# Count all variants, excluding header lines, handling compressed files
if [[ "${VCF_FILE}" == *.gz ]]; then
VARIANT_COUNT=$(zgrep -v '^#' "${VCF_FILE}" | wc -l)
else
VARIANT_COUNT=$(grep -v '^#' "${VCF_FILE}" | wc -l)
fi
echo "Total variants in ${VCF_FILE}: ${VARIANT_COUNT}"

**Try it yourself:** Modify the code above or write your own version

In [ ]:
%%bash
# TODO: Run your bash commands here
# Hint: Try modifying the example above




> **Important:** Notice the use of `>&2` when printing error messages. This redirects the message to the "standard error" stream instead of "standard output." This is vital for separating error notifications from the script's actual results, allowing other programs (or you!) to process the output without having to parse out error text.

**Expected Output (if file exists):**
```text
Total variants in my_variants.vcf.gz: 12345
```

**Expected Output (if argument is missing):**
```text
./your_script.sh: Error: Missing input file. Usage: ./your_script.sh <variants.vcf[.gz]>
```

### 3.2 Automate Batch Processing with Loops

Use a `for` loop to apply the same analysis to multiple files. This script iterates through all FASTQ files in a directory to generate a summary report.

In [ ]:
%%bash
#!/bin/bash
set -euo pipefail
shopt -s nullglob
# --- Configuration ---
# Usage: $0 [INPUT_DIRECTORY]
INPUT_DIR=${1:-.} # Default to current directory
SUMMARY_FILE="read_count_summary.csv"
FILES=("$INPUT_DIR"/*.fastq.gz)
# Check if any files were found
if [ ${#FILES[@]} -eq 0 ]; then
echo "Warning: No .fastq.gz files found in '$INPUT_DIR'." >&2
exit 0
fi
echo "filename,read_count" > "${SUMMARY_FILE}"
for file in "${FILES[@]}"; do
COUNT=$(( $(zcat "${file}" | wc -l) / 4 ))
echo "$(basename "${file}"),${COUNT}" >> "${SUMMARY_FILE}"
done
echo "Batch processing complete. See ${SUMMARY_FILE}"

**Try it yourself:** Modify the code above or write your own version

In [ ]:
%%bash
# TODO: Run your bash commands here
# Hint: Try modifying the example above




> **Medical Background:** A standard FASTQ file contains four lines per DNA read: 1) a sequence identifier, 2) the raw sequence data, 3) a separator line, and 4) the quality scores for the sequence. By counting the total lines and dividing by four, we get an accurate count of the reads in the file. This 4-line structure is a widely adopted standard in genomics (Cock et al., 2010).

> **In Practice:** While this script is excellent for learning, production-level bioinformatics workflows often use highly optimized, compiled tools like `seqtk` or `fastp` for QC tasks. These tools are significantly faster and more robust for handling the massive datasets generated by modern sequencers. Our script demonstrates the underlying logic, which is a valuable skill for building custom analysis pipelines.

**Expected Output (in the terminal):**
```text
Batch processing complete. See read_count_summary.csv
```

**Expected content of `read_count_summary.csv`:**
```csv
filename,read_count
sample_A.fastq.gz,2500000
sample_B.fastq.gz,2480000
```

---

## 4. Practice Exercises

### Exercise 1: Count Reads in a Single FASTQ File

**Objective:** Write a simple script to practice basic file I/O and arithmetic in Bash.
**Time:** 5 minutes
**Medical Context:** Performing a quick, preliminary quality check on a raw sequencing file from a patient sample to ensure it contains data before running a full analysis pipeline.

Create a script named `count_reads.sh` that accepts one argument: the path to a gzipped FASTQ file. The script should calculate the number of reads and print the result in the format: `File [filename] contains [number] reads.`

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```bash
#!/bin/bash
set -euo pipefail

# Usage: ./count_reads.sh <file.fastq.gz>
FASTQ_FILE=${1:?"Error: Missing input FASTQ file."}

if [ ! -f "${FASTQ_FILE}" ]; then
    echo "Error: File not found: ${FASTQ_FILE}" >&2
    exit 1
fi

# A FASTQ file has 4 lines per read
LINE_COUNT=$(zcat "${FASTQ_FILE}" | wc -l)
READ_COUNT=$(( LINE_COUNT / 4 ))

echo "File $(basename "${FASTQ_FILE}") contains ${READ_COUNT} reads."
```
**Explanation:** The script validates that a file argument was provided and that the file exists. It then uses `zcat` to decompress the file and pipes the output to `wc -l` to count lines. Finally, it uses Bash arithmetic `((...))` to divide by 4 to get the read count.
**Key Learning:** Using command-line arguments, performing file checks, and using arithmetic expansion.


</div>
</details>

### Exercise 2: Extract Variant Positions

**Objective:** Practice using pipes to chain `grep`, `cut`, and `tee` for data extraction.
**Time:** 10 minutes
**Medical Context:** A genomics researcher needs to quickly isolate all variant positions on chromosome 22 to compare them against a known list of disease-associated markers. To help them, write a reusable script named `extract_positions.sh`.

Your script must accept a VCF file, verify it exists, and extract the variant positions (column 2) for chromosome 22 only. Save the output to `chr22_positions.txt` and print a confirmation with the total count, like: `Extracted 150 positions to chr22_positions.txt`.

**Hint:** You can create a pipeline using `grep` to filter for lines starting with the chromosome 22 identifier (e.g., `'^22\t'` or `'^chr22\t'`) and `cut` to extract the second column. To both save the output to a file *and* count the lines in a single command, investigate how the `tee` command can be used within a pipeline.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```bash
#!/bin/bash
set -euo pipefail

VCF_FILE=${1:?"Error: Missing input VCF file."}

if [ ! -f "${VCF_FILE}" ]; then
    echo "Error: File not found: ${VCF_FILE}" >&2
    exit 1
fi

# Use grep for chr22, cut for column 2, and tee to write to file and pipe to wc
COUNT=$(grep -v '^#' "${VCF_FILE}" | grep -E '^(chr)?22\t' | cut -f2 | tee chr22_positions.txt | wc -l)

echo "Extracted ${COUNT} positions to chr22_positions.txt"
```
**Explanation:** The script first filters out header lines (`grep -v '^#'`), then finds lines for chromosome 22. `cut -f2` extracts the position. The `tee` command writes the positions to the file *and* passes them to `wc -l` to be counted.
**Key Learning:** Combining multiple commands in a pipeline and using `tee` to split output.


</div>
</details>

### Exercise 3: General-Purpose Variant Extraction Script

**Objective:** Write a flexible and reusable script that accepts multiple arguments and handles different file types.
**Time:** 15 minutes
**Medical Context:** Building a reusable tool for a genomics core facility. Researchers need to extract variants for different chromosomes from both compressed and uncompressed files.

Create `extract_variants_by_chr.sh`. It must accept two arguments: 1) the VCF file (which can be `.vcf` or `.vcf.gz`) and 2) the chromosome number. The output file should be named dynamically, e.g., `chr22_positions.txt`.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```bash
#!/bin/bash
set -euo pipefail

# --- Configuration ---
# Usage: $0 <variants.vcf[.gz]> <chromosome>
VCF_FILE=${1:?"Error: Missing input VCF file. Usage: $0 <variants.vcf[.gz]>"}
CHROMOSOME=${2:?"Error: Missing chromosome argument."}
OUTPUT_FILE="chr${CHROMOSOME}_positions.txt"

if [ ! -f "${VCF_FILE}" ]; then
    echo "Error: Input file not found: ${VCF_FILE}" >&2
    exit 1
fi

# --- Workflow ---
# Determine the correct grep tool for compressed or uncompressed files
if [[ "${VCF_FILE}" == *.gz ]]; then
    GREP_CMD="zgrep"
else
    GREP_CMD="grep"
fi

# Extract positions, save them to a file, and count them
COUNT=$($GREP_CMD -v '^#' "${VCF_FILE}" | grep -E "^(chr)?${CHROMOSOME}\t" | cut -f2 | tee "${OUTPUT_FILE}" | wc -l)

echo "Extracted ${COUNT} positions for chromosome ${CHROMOSOME} to ${OUTPUT_FILE}"
```
**Explanation:** This robust script takes two arguments, performs checks, and dynamically sets the `GREP_CMD` based on the file extension. The `grep` pattern and output filename both use the `${CHROMOSOME}` variable, making the script flexible and reusable.
**Key Learning:** Creating flexible scripts with multiple arguments, dynamic variable assignment, and conditional logic.


</div>
</details>

---

## 5. Practical Applications

*   **Automated NGS Quality Control (QC):** In clinical labs, a bash script often automates the first QC step. It loops through hundreds of patient FASTQ files from a sequencing run, runs a tool like FastQC on each, and aggregates key metrics into a single summary report, immediately flagging low-quality samples that could compromise a diagnosis.
*   **High-Throughput Variant Annotation:** After a VCF file is generated, a script automates the critical next step: annotation. The script takes the VCF file as input and uses tools like SnpEff to cross-reference each variant against databases like ClinVar, adding information on known disease associations to aid clinical interpretation.
*   **Cancer Genomics Somatic vs. Germline Filtering:** When analyzing tumor-normal pairs, a script can automate the identification of tumor-specific (somatic) mutations. It compares the VCF files from the tumor and normal tissue, using command-line tools to create a filtered list of somatic mutations that are the primary targets for cancer research and targeted therapies.

---

## 6. Summary and Key Takeaways

In this section, we've explored how to write Bash scripts to automate bioinformatics workflows, moving from single commands to powerful, reproducible analysis pipelines. You learned to build robust scripts that can handle errors and process large batches of data efficiently, skills that are fundamental to modern computational biology.

*   **Build Robustly:** Always start scripts with `set -euo pipefail` to prevent silent failures and ensure data integrity.
*   **Validate Inputs:** Check for the existence of required arguments and input files at the beginning of your script to avoid errors during execution.
*   **Automate with Loops:** Use `for` loops to iterate over collections of files, making your analysis scalable from one sample to thousands.
*   **Handle File Types:** Write flexible scripts that can process different file formats, such as compressed (`.gz`) and uncompressed files, by using conditional logic.
*   **Chain Commands with Pipes:** Create powerful data processing workflows by piping the output of one tool into the input of another.

> **Reflection Moment:**
> The batch processing script performs a simple count. How would you modify its `for` loop to run a more complex, multi-step analysis on each FASTQ file? For example, what if you needed to run a quality control tool, then an aligner, and finally a variant caller on each sample?

---


---

## 📝 Interactive Practice

Practice the concepts with these interactive exercises:

### corrupted data or incorrect scientific conclusions. It forces the script to stop immediately if a command fails (`-e`), a variable is unset (`-u`), or a command in a pipeline fails (`-o pipefail`).

In [ ]:
%%bash
#!/bin/bash
set -euo pipefail
# --- Configuration ---
# Usage: $0 <variants.vcf[.gz]>
VCF_FILE=${1:?"Error: Missing input file. Usage: $0 <variants.vcf[.gz]>"}
if [ ! -f "${VCF_FILE}" ]; then
echo "Error: Input file not found: ${VCF_FILE}" >&2
exit 1
fi
# --- Workflow ---
# Count all variants, excluding header lines, handling compressed files
if [[ "${VCF_FILE}" == *.gz ]]; then
VARIANT_COUNT=$(zgrep -v '^#' "${VCF_FILE}" | wc -l)
else
VARIANT_COUNT=$(grep -v '^#' "${VCF_FILE}" | wc -l)
fi
echo "Total variants in ${VCF_FILE}: ${VARIANT_COUNT}"

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### Use a `for` loop to apply the same analysis to multiple files. This script iterates through all FASTQ files in a directory to generate a summary report.

In [ ]:
%%bash
#!/bin/bash
set -euo pipefail
shopt -s nullglob
# --- Configuration ---
# Usage: $0 [INPUT_DIRECTORY]
INPUT_DIR=${1:-.} # Default to current directory
SUMMARY_FILE="read_count_summary.csv"
FILES=("$INPUT_DIR"/*.fastq.gz)
# Check if any files were found
if [ ${#FILES[@]} -eq 0 ]; then
echo "Warning: No .fastq.gz files found in '$INPUT_DIR'." >&2
exit 0
fi
echo "filename,read_count" > "${SUMMARY_FILE}"
for file in "${FILES[@]}"; do
COUNT=$(( $(zcat "${file}" | wc -l) / 4 ))
echo "$(basename "${file}"),${COUNT}" >> "${SUMMARY_FILE}"
done
echo "Batch processing complete. See ${SUMMARY_FILE}"

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### <summary>Solution</summary>

In [ ]:
%%bash
#!/bin/bash
set -euo pipefail
# Usage: ./count_reads.sh <file.fastq.gz>
FASTQ_FILE=${1:?"Error: Missing input FASTQ file."}
if [ ! -f "${FASTQ_FILE}" ]; then
echo "Error: File not found: ${FASTQ_FILE}" >&2
exit 1
fi
# A FASTQ file has 4 lines per read
LINE_COUNT=$(zcat "${FASTQ_FILE}" | wc -l)
READ_COUNT=$(( LINE_COUNT / 4 ))
echo "File $(basename "${FASTQ_FILE}") contains ${READ_COUNT} reads."

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### <summary>Solution</summary>

In [ ]:
%%bash
#!/bin/bash
set -euo pipefail
VCF_FILE=${1:?"Error: Missing input VCF file."}
if [ ! -f "${VCF_FILE}" ]; then
echo "Error: File not found: ${VCF_FILE}" >&2
exit 1
fi
# Use grep for chr22, cut for column 2, and tee to write to file and pipe to wc
COUNT=$(grep -v '^#' "${VCF_FILE}" | grep -E '^(chr)?22\t' | cut -f2 | tee chr22_positions.txt | wc -l)
echo "Extracted ${COUNT} positions to chr22_positions.txt"

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above




### <summary>Solution</summary>

In [ ]:
%%bash
#!/bin/bash
set -euo pipefail
# --- Configuration ---
# Usage: $0 <variants.vcf[.gz]> <chromosome>
VCF_FILE=${1:?"Error: Missing input VCF file. Usage: $0 <variants.vcf[.gz]>"}
CHROMOSOME=${2:?"Error: Missing chromosome argument."}
OUTPUT_FILE="chr${CHROMOSOME}_positions.txt"
if [ ! -f "${VCF_FILE}" ]; then
echo "Error: Input file not found: ${VCF_FILE}" >&2
exit 1
fi
# --- Workflow ---
# Determine the correct grep tool for compressed or uncompressed files
if [[ "${VCF_FILE}" == *.gz ]]; then
GREP_CMD="zgrep"
else
GREP_CMD="grep"
fi
# Extract positions, save them to a file, and count them
COUNT=$($GREP_CMD -v '^#' "${VCF_FILE}" | grep -E "^(chr)?${CHROMOSOME}\t" | cut -f2 | tee "${OUTPUT_FILE}" | wc -l)
echo "Extracted ${COUNT} positions for chromosome ${CHROMOSOME} to ${OUTPUT_FILE}"

### Try It Yourself:
Modify the code above or write your own version:

In [ ]:
%%bash
# TODO: Write your bash commands here
# Hint: Try modifying the example above


